In [1]:
#%pip install "numpy<2.0.0" "pandas<3.0.0" "torch==2.2.2" "transformers>=4.41.0,<4.45.0" "sentence-transformers>=3.0.0,<4.0.0"

In [2]:
#streamlit run app.py

In [3]:
import pandas as pd
import lyricsgenius
import time
import os
import tarfile
from dotenv import load_dotenv

In [4]:
load_dotenv("../.env")
genius = lyricsgenius.Genius(
    os.getenv("GENIUS_ACCESS_TOKEN"), 
    timeout=15, 
    retries=3, 
    sleep_time=2.0 
)

In [5]:
tags_file = "../data/processed/lastfm_extracted_tags.pkl"
tags_df = pd.read_pickle(tags_file)
existing_df = pd.read_csv("final_df.csv", index_col=0)

In [6]:
if os.path.exists("failed_tracks.csv"):
    failed_df = pd.read_csv("failed_tracks.csv")
    failed_keys = failed_df['match_key'].tolist()
else:
    failed_keys = []

tags_df['match_key'] = tags_df['artist'].astype(str).str.lower() + " - " + tags_df['title'].astype(str).str.lower()
existing_df['match_key'] = existing_df['artist'].astype(str).str.lower() + " - " + existing_df['song'].astype(str).str.lower()

unprocessed_df = tags_df[~tags_df['match_key'].isin(existing_df['match_key'])].copy()
unprocessed_df = unprocessed_df[~unprocessed_df['match_key'].isin(failed_keys)]
unprocessed_df = unprocessed_df.drop(columns=['match_key'])

print(f"Total remaining tracks to process: {len(unprocessed_df)}")

Total remaining tracks to process: 976350


In [7]:
BATCH_SIZE = 500
batch_df = unprocessed_df.head(BATCH_SIZE).copy()
new_records = []
failed_records = []

print(f"Starting batch extraction for {BATCH_SIZE} tracks...")

for index, row in batch_df.iterrows():
    match_key = str(row['artist']).lower() + " - " + str(row['title']).lower()
    try:
        song = genius.search_song(row['title'], row['artist'])
        
        if song and song.lyrics:
            new_records.append({
                'artist': row['artist'],
                'song': row['title'],
                'link': song.url,
                'text': song.lyrics,
                'clean_artist': str(row['artist']).lower(),
                'clean_title': str(row['title']).lower(),
                'tags': row['tags'],
                'genres': [] 
            })
        else:
            failed_records.append({
                'match_key': match_key,
                'artist': row['artist'],
                'song': row['title'],
                'link': '',
                'text': '',
                'clean_artist': str(row['artist']).lower(),
                'clean_title': str(row['title']).lower(),
                'tags': row['tags'],
                'genres': [] 
            })
            
        time.sleep(2.0) 
        
    except Exception as e:
        print(f"API Error on {row['artist']} - {row['title']}: {e}")
        time.sleep(5) 

if new_records:
    append_df = pd.DataFrame(new_records)
    append_df.to_csv("final_df.csv", mode='a', header=False, index=True)
    print(f"Successfully appended {len(append_df)} tracks to final_df.csv.")
else:
    print("No valid lyrics found in this batch.")

if failed_records:
    failed_df_append = pd.DataFrame(failed_records)
    write_header = not os.path.exists("failed_tracks.csv")
    failed_df_append.to_csv("failed_tracks.csv", mode='a', header=write_header, index=False)
    print(f"Blacklisted {len(failed_records)} unfindable tracks.")

Starting batch extraction for 500 tracks...


Successfully appended 281 tracks to final_df.csv.
Blacklisted 219 unfindable tracks.
